In [1]:
import os

os.chdir('..')
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # set for determinism

## Loading the Dataset:

In this section the pointcloud is loaded. The SIREN paper suggests normalizing the point coordinates as periodic activations implicitly expect a bounded input. 

In [2]:
import open3d as o3d
import numpy as np
import torch
import matplotlib.pyplot as plt
import src.model.SIREN as si
from src.model.training import train
import src.loss.SDF_loss as loss
from src.mesh_extraction.marching_cubes_test import write_obj
import src.model.MLP as simple
import src.data.dataset as data
import src.model.pruning_module as pm
import src.mesh_extraction.marching_cubes_gpu as marching_cubes
import random

def set_seeds():
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

mesh = data.MeshDataset('data/pointclouds/Armadillo/Armadillo.ply')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Defining the Model

In this cell we will define the SIREN model. This particular INR uses sine activations for nonlinearity and is supposed to capture more information given the underlying data when compared to a model that uses ReLU activations. This way, a good INR accuracy can be achieved with fewer neurons.

In [3]:
size_per_layer = 256
set_seeds()
model_no = si.SIRENSDF()
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=0.25, lambda_normal=2, lambda_inter=0.05, lambda_off=2, lambda_twd=1e-4, model=model_no)
optimizer = torch.optim.Adam(model_no.parameters(), lr=1e-4)
si.weight_stats(model_no)

Weight statistics per layer:
------------------------------------------------------------
Hidden layer  0
  Weight: mean=7.1341e-03, std=1.8916e-01, min=-3.3078e-01, max=3.3230e-01
  Bias  : mean=-1.2152e-02, std=1.8760e-01, min=-3.2975e-01, max=3.3306e-01
  Omega scale (exp): mean=1.0000e+00, std=0.0000e+00
------------------------------------------------------------
Hidden layer  1
  Weight: mean=6.2948e-06, std=2.9461e-03, min=-5.1030e-03, max=5.1029e-03
  Bias  : mean=3.2051e-05, std=2.9077e-03, min=-5.0769e-03, max=5.0955e-03
------------------------------------------------------------
Hidden layer  2
  Weight: mean=-2.4295e-05, std=2.9552e-03, min=-5.1031e-03, max=5.1029e-03
  Bias  : mean=-4.6270e-05, std=2.9337e-03, min=-5.0569e-03, max=5.1006e-03
------------------------------------------------------------
Hidden layer  3
  Weight: mean=-1.7248e-05, std=2.9479e-03, min=-5.1030e-03, max=5.1028e-03
  Bias  : mean=1.3535e-04, std=2.7929e-03, min=-4.9978e-03, max=5.0838e-03
------

/home/nikola/adaptive_pruning_SIREN/src/model/SIREN.py:117: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1823.)
  print(f"  Bias  : mean={b.mean():.4e}, std={b.std():.4e}, min={b.min():.4e}, max={b.max():.4e}")


## Model training without pruning or densification



In [4]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_no, loss=model_loss, optimizer=optimizer, scene=mesh.scene)

Step 0 | Loss 2.7384209632873535
Step 10 | Loss 0.5176354646682739
Step 20 | Loss 0.4287896454334259
Step 30 | Loss 0.38053232431411743
Step 40 | Loss 0.3563810884952545
Step 50 | Loss 0.3320784866809845
Step 60 | Loss 0.3231888711452484
Step 70 | Loss 0.30770638585090637
Step 80 | Loss 0.3001760244369507
Step 90 | Loss 0.29540130496025085
Step 100 | Loss 0.28889572620391846
Step 110 | Loss 0.2869040369987488
Step 120 | Loss 0.2792885899543762
Step 130 | Loss 0.27895158529281616
Step 140 | Loss 0.2750289738178253
Step 150 | Loss 0.27495652437210083
Step 160 | Loss 0.2684859335422516
Step 170 | Loss 0.25759997963905334
Step 180 | Loss 0.26309117674827576
Step 190 | Loss 0.25888386368751526
Step 200 | Loss 0.26008978486061096
Step 210 | Loss 0.2583893835544586
Step 220 | Loss 0.2535340189933777
Step 230 | Loss 0.2517163157463074
Step 240 | Loss 0.24871374666690826
Step 250 | Loss 0.24622920155525208
Step 260 | Loss 0.24386952817440033
Step 270 | Loss 0.2436744123697281
Step 280 | Loss 0.

#### Model size after pruning

In [5]:
model_no.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  256 neurons
Hidden layer  1:  256 neurons
Hidden layer  2:  256 neurons
Hidden layer  3:  256 neurons
Hidden layer  4:  256 neurons
Final layer    :    1 neurons


In [6]:
marching_cubes.write_obj("armadillo_128_unpruned.obj", model=model_no, resolution=128, level=0.0)

## Model training with densification

In [4]:
set_seeds()
model_yes = si.SIRENSDF()
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=0.25, lambda_normal=2, lambda_inter=0.05, lambda_off=2, lambda_twd=1e-4, model=model_yes)
optimizer = torch.optim.Adam(model_yes.parameters(), lr=1e-4)
si.weight_stats(model_yes)

Weight statistics per layer:
------------------------------------------------------------
Hidden layer  0
  Weight: mean=7.1341e-03, std=1.8916e-01, min=-3.3078e-01, max=3.3230e-01
  Bias  : mean=-1.2152e-02, std=1.8760e-01, min=-3.2975e-01, max=3.3306e-01
  Omega scale (exp): mean=1.0000e+00, std=0.0000e+00
------------------------------------------------------------
Hidden layer  1
  Weight: mean=6.2948e-06, std=2.9461e-03, min=-5.1030e-03, max=5.1029e-03
  Bias  : mean=3.2051e-05, std=2.9077e-03, min=-5.0769e-03, max=5.0955e-03
------------------------------------------------------------
Hidden layer  2
  Weight: mean=-2.4295e-05, std=2.9552e-03, min=-5.1031e-03, max=5.1029e-03
  Bias  : mean=-4.6270e-05, std=2.9337e-03, min=-5.0569e-03, max=5.1006e-03
------------------------------------------------------------
Hidden layer  3
  Weight: mean=-1.7248e-05, std=2.9479e-03, min=-5.1030e-03, max=5.1028e-03
  Bias  : mean=1.3535e-04, std=2.7929e-03, min=-4.9978e-03, max=5.0838e-03
------

In [5]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_yes, loss=model_loss, optimizer=optimizer, scene=mesh.scene, densification=True)

Step 0 | Loss 2.7384209632873535
Step 10 | Loss 0.5176354646682739
Step 20 | Loss 0.4287896454334259
Step 30 | Loss 0.38053232431411743
Step 40 | Loss 0.3563810884952545
Step 50 | Loss 0.3320784866809845
Step 60 | Loss 0.3231888711452484
Step 70 | Loss 0.30770638585090637
Step 80 | Loss 0.3001760244369507
Step 90 | Loss 0.29540130496025085
Step 100 | Loss 0.28889572620391846
Step 110 | Loss 0.2869040369987488
Step 120 | Loss 0.2792885899543762
Step 130 | Loss 0.27895158529281616
Step 140 | Loss 0.2750289738178253
Step 150 | Loss 0.27495652437210083
Step 160 | Loss 0.2684859335422516
Step 170 | Loss 0.25759997963905334
Step 180 | Loss 0.26309117674827576
Step 190 | Loss 0.25888386368751526
Added 52 frequencies to the embedding layer.
Step 200 | Loss 0.26008978486061096
Step 210 | Loss 0.3215702772140503
Step 220 | Loss 0.2839159667491913
Step 230 | Loss 0.26066917181015015
Step 240 | Loss 0.24549925327301025
Step 250 | Loss 0.23705948889255524
Step 260 | Loss 0.2310156524181366
Step 270

In [6]:
model_yes.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  308 neurons
Hidden layer  1:  256 neurons
Hidden layer  2:  256 neurons
Hidden layer  3:  256 neurons
Hidden layer  4:  256 neurons
Final layer    :    1 neurons


In [7]:
marching_cubes.write_obj("armadillo_128_densified.obj", model=model_yes, resolution=128, level=0.0)

## AIRe: Model training (no densification)

In [5]:
set_seeds()
model_aire_no = si.SIRENSDF()
prune_AIRe = pm.AIRe(model_aire_no, 0.4, int(size_per_layer * 0.15))
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=0.25, lambda_normal=2, lambda_inter=0.05, lambda_off=2, lambda_twd=1e-4, model=model_aire_no, pruning_module=prune_AIRe)
optimizer = torch.optim.Adam(model_aire_no.parameters(), lr=1e-4)

In [6]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_aire_no, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_AIRe)

Step 0 | Loss 2.7478036880493164
Step 10 | Loss 0.5270693898200989
Step 20 | Loss 0.4382750391960144
Step 30 | Loss 0.3900386393070221
Step 40 | Loss 0.36589881777763367
Step 50 | Loss 0.3415887653827667
Step 60 | Loss 0.33271926641464233
Step 70 | Loss 0.31726130843162537
Step 80 | Loss 0.3097176253795624
Step 90 | Loss 0.3049064874649048
Step 100 | Loss 0.2983993887901306
Step 110 | Loss 0.2964012622833252
Step 120 | Loss 0.28876978158950806
Step 130 | Loss 0.2884427607059479
Step 140 | Loss 0.2845064103603363
Step 150 | Loss 0.2843857407569885
Step 160 | Loss 0.277934730052948
Step 170 | Loss 0.26705682277679443
Step 180 | Loss 0.27254366874694824
Step 190 | Loss 0.2683834731578827
Step 200 | Loss 0.2695077359676361
Step 210 | Loss 0.2677944600582123
Step 220 | Loss 0.26300033926963806
Step 230 | Loss 0.26117193698883057
Step 240 | Loss 0.25813886523246765
Step 250 | Loss 0.2556002736091614
Step 260 | Loss 0.2532746195793152
Step 270 | Loss 0.25310903787612915
Step 280 | Loss 0.2504

In [8]:
model_aire_no.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  256 neurons
Hidden layer  1:  153 neurons
Hidden layer  2:  153 neurons
Hidden layer  3:  153 neurons
Hidden layer  4:  153 neurons
Final layer    :    1 neurons


In [7]:
marching_cubes.write_obj("armadillo_128_AIRe.obj", model=model_aire_no, resolution=128, level=0.0)

## DepGraph: Model training (no densification)

In [4]:
set_seeds()
model_dep_no = si.SIRENSDF()
prune_DepGraph = pm.DepGraph(model_dep_no, 0.4, int(size_per_layer * 0.15))
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=0.25, lambda_normal=2, lambda_inter=0.05, lambda_off=2, lambda_twd=1e-4, model=model_dep_no, pruning_module=prune_DepGraph)
optimizer = torch.optim.Adam(model_dep_no.parameters(), lr=1e-4)

In [5]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_dep_no, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_DepGraph)

Step 0 | Loss 2.7396719455718994
Step 10 | Loss 0.5188050270080566
Step 20 | Loss 0.4299154281616211
Step 30 | Loss 0.38161495327949524
Step 40 | Loss 0.3574238717556
Step 50 | Loss 0.33308857679367065
Step 60 | Loss 0.3241972327232361
Step 70 | Loss 0.30872538685798645
Step 80 | Loss 0.301169216632843
Step 90 | Loss 0.29632946848869324
Step 100 | Loss 0.28981319069862366
Step 110 | Loss 0.2878071367740631
Step 120 | Loss 0.2802317440509796
Step 130 | Loss 0.27985459566116333
Step 140 | Loss 0.27598127722740173
Step 150 | Loss 0.2758615016937256
Step 160 | Loss 0.2694056034088135
Step 170 | Loss 0.2585516571998596
Step 180 | Loss 0.264021635055542
Step 190 | Loss 0.25982585549354553
Step 200 | Loss 0.26101070642471313
Step 210 | Loss 0.25935113430023193
Step 220 | Loss 0.25453391671180725
Step 230 | Loss 0.25269752740859985
Step 240 | Loss 0.24970144033432007
Step 250 | Loss 0.24716567993164062
Step 260 | Loss 0.24486802518367767
Step 270 | Loss 0.24465976655483246
Step 280 | Loss 0.24

In [6]:
model_dep_no.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  256 neurons
Hidden layer  1:  153 neurons
Hidden layer  2:  153 neurons
Hidden layer  3:  153 neurons
Hidden layer  4:  153 neurons
Final layer    :    1 neurons


In [7]:
marching_cubes.write_obj("armadillo_128_DepGraph.obj", model=model_dep_no, resolution=128, level=0.0)

## AIRe: Model training with densification

In [4]:
set_seeds()
model_aire_yes = si.SIRENSDF()
prune_AIRe = pm.AIRe(model_aire_yes, 0.4, int(size_per_layer * 0.15))
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=0.25, lambda_normal=2, lambda_inter=0.05, lambda_off=2, lambda_twd=1e-4, model=model_aire_yes, pruning_module=prune_AIRe)
optimizer = torch.optim.Adam(model_aire_yes.parameters(), lr=1e-4)

In [5]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_aire_yes, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_AIRe, densification=True)

Step 0 | Loss 2.7478036880493164
Step 10 | Loss 0.5270693898200989
Step 20 | Loss 0.4382750391960144
Step 30 | Loss 0.3900386393070221
Step 40 | Loss 0.36589881777763367
Step 50 | Loss 0.3415887653827667
Step 60 | Loss 0.33271926641464233
Step 70 | Loss 0.31726130843162537
Step 80 | Loss 0.3097176253795624
Step 90 | Loss 0.3049064874649048
Step 100 | Loss 0.2983993887901306
Step 110 | Loss 0.2964012622833252
Step 120 | Loss 0.28876978158950806
Step 130 | Loss 0.2884427607059479
Step 140 | Loss 0.2845064103603363
Step 150 | Loss 0.2843857407569885
Step 160 | Loss 0.277934730052948
Step 170 | Loss 0.26705682277679443
Step 180 | Loss 0.27254366874694824
Step 190 | Loss 0.2683834731578827
Added 52 frequencies to the embedding layer.
Step 200 | Loss 0.2695077359676361
Step 210 | Loss 0.32724684476852417
Step 220 | Loss 0.2906937897205353
Step 230 | Loss 0.267962783575058
Step 240 | Loss 0.2524513006210327
Step 250 | Loss 0.24446450173854828
Step 260 | Loss 0.2384418249130249
Step 270 | Loss

In [6]:
model_aire_yes.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  308 neurons
Hidden layer  1:  153 neurons
Hidden layer  2:  153 neurons
Hidden layer  3:  153 neurons
Hidden layer  4:  153 neurons
Final layer    :    1 neurons


In [7]:
marching_cubes.write_obj("armadillo_128_AIRe_densified.obj", model=model_aire_yes, resolution=128, level=0.0)

## DepGraph: Model training with densification

In [4]:
set_seeds()
model_dep_yes = si.SIRENSDF()
prune_DepGraph = pm.DepGraph(model_dep_yes, 0.4, int(0.15*size_per_layer))
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=0.25, lambda_normal=2, lambda_inter=0.05, lambda_off=2, lambda_twd=1e-4, model=model_dep_yes, pruning_module=prune_DepGraph)
optimizer = torch.optim.Adam(model_dep_yes.parameters(), lr=1e-4)

In [5]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_dep_yes, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_DepGraph, densification=True)

Step 0 | Loss 2.7396719455718994
Step 10 | Loss 0.5188050270080566
Step 20 | Loss 0.4299154281616211
Step 30 | Loss 0.38161495327949524
Step 40 | Loss 0.3574238717556
Step 50 | Loss 0.33308857679367065
Step 60 | Loss 0.3241972327232361
Step 70 | Loss 0.30872538685798645
Step 80 | Loss 0.301169216632843
Step 90 | Loss 0.29632946848869324
Step 100 | Loss 0.28981319069862366
Step 110 | Loss 0.2878071367740631
Step 120 | Loss 0.2802317440509796
Step 130 | Loss 0.27985459566116333
Step 140 | Loss 0.27598127722740173
Step 150 | Loss 0.2758615016937256
Step 160 | Loss 0.2694056034088135
Step 170 | Loss 0.2585516571998596
Step 180 | Loss 0.264021635055542
Step 190 | Loss 0.25982585549354553
Added 52 frequencies to the embedding layer.
Step 200 | Loss 0.26101070642471313
Step 210 | Loss 0.32208436727523804
Step 220 | Loss 0.2847960889339447
Step 230 | Loss 0.26170867681503296
Step 240 | Loss 0.24650166928768158
Step 250 | Loss 0.2379319667816162
Step 260 | Loss 0.23199363052845
Step 270 | Loss 

In [6]:
model_dep_yes.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  308 neurons
Hidden layer  1:  153 neurons
Hidden layer  2:  153 neurons
Hidden layer  3:  153 neurons
Hidden layer  4:  153 neurons
Final layer    :    1 neurons


In [7]:
marching_cubes.write_obj("armadillo_128_DepGraph_densified.obj", model=model_dep_yes, resolution=128, level=0.0)

In [24]:
#from src.mesh_extraction import marching_cubes_test 
#marching_cubes_test.write_obj("lucy_128_gt.obj", mesh.scene, 128, 0.0)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Sample a few surface points and check their SDF values
test_points = mesh.vertices[:10]  # First 10 points
test_tensor = torch.from_numpy(test_points).float().to(device)
with torch.no_grad():
    sdf_values = model_no(test_tensor)
print("SDF values for surface points:")
print(sdf_values)
print("Mean absolute SDF:", torch.abs(sdf_values).mean().item())

SDF values for surface points:
tensor([[ 2.1020e-05],
        [ 3.4788e-03],
        [-2.3386e-04],
        [ 3.9918e-05],
        [ 4.1725e-03],
        [ 4.5397e-03],
        [ 4.0368e-03],
        [-4.1481e-06],
        [ 1.9721e-03],
        [-3.3731e-03]], device='cuda:0')
Mean absolute SDF: 0.002187199890613556


In [ ]:
# import src.mesh_extraction.marching_cubes_test as marching_cubes_test
# marching_cubes_test.write_obj("armadillo_mesh_ground_truth_128.obj", scene=mesh.scene, resolution=128, level=0.0)


In [ ]:
# Make batched point
test_point = torch.from_numpy(np.array([[-1, -1, -1]])).float().to(device)

# Compute SIREN model prediction
sdf_pred = model_no(test_point)
print("SIREN prediction:", sdf_pred)

# Compute Open3D signed distance
distance = mesh.scene.compute_signed_distance(
    o3d.core.Tensor(test_point.cpu().numpy(), dtype=o3d.core.Dtype.Float32)
)
print("Open3D SDF:", distance.numpy())

SIREN prediction: tensor([[0.1634]], device='cuda:0', grad_fn=<AddmmBackward0>)
Open3D SDF: [0.96991843]


In [25]:
torch.save(model_no.state_dict(), "model_no.pth")
torch.save(model_yes.state_dict(), "model_yes.pth")
torch.save(model_aire_no.state_dict(), "model_aire_no.pth")
torch.save(model_dep_no.state_dict(), "model_dep_no.pth")
torch.save(model_aire_yes.state_dict(), "model_aire_yes.pth")
torch.save(model_dep_yes.state_dict(), "model_dep_yes.pth")